# **Source-Photometry Lephare Identification Tool (SPLIT) for star-galaxy separation**

This is the notebook where we concatenate the whole functioning of SPLIT, an addons developped in parallel of LePHARE too . To run its whole pipeline, it takes in input the $\chi^2$ full library of LePhare (fullChi2). Hence there is two codes to run, divided in several steps each. If you don't have fullChi2, first you have to run LePHARE separately from SPLIT.
- **LePHARE**:
    - 1 - Build filter library.
    - 2 - Build sed library.
    - 3 - Build synthetic magnitude library.
    - 4 - Run photo-z

An example of command lines to use to run LePHARE from step 1 to 4 is avalaible in the StarGal folder of this repository in *stargal/LePHARE_for_SPLIT.txt*.


Then you can run SPLIT, in this notebook to control the steps and plot, or in script (*stargal/scripts/SPLIT.py*). In its core, it performs three main steps, which can be use independentely, depending on how efficient you want your stargal to be, and what do you have as inputs. It takes in input:
- **SPLIT**:
    - 1. $\chi^2_{best fit}$ comparison. Compare the best likelihood value from each library ($\chi^2_{star}$, $\chi^2_{gal}$) and compute the Bayesian Information Criterion (BIC): $\mathrm{BIC}_{m}=\chi^2_{m}+k_m\ln(n_{data}) \Rightarrow \mathrm{\Delta BIC}_{m,j}=\Delta\chi^2_{m,j}+(k_m-k_j)\ln(n_{data})$. Where $\Delta\chi^2_{m,j}=\chi^2_{m}-\chi^2_{j}$, $k_{m,j}$ is the number of free parameters in models {$m,j$} and $n_{data}$ is the number of observations for the source, i.e. $n_{bands}$ in our case. If $\Delta \mathrm{BIC} < 0$, model m is preferred, else it is model j. This is better than doing a direct $\chi^2_{m,j}$ comparison since the two models are non-nested in our case. Note that $\Delta \mathrm{BIC}$ is an approximation of $C_\mathrm{Bayes}$ for high $S/N$. It only requires $\chi^2_{best fit}$ of each models. 
    - 2. $\chi^2_{best fit}$ comparison. $C_\mathrm{Bayes} = 2\ln(B_{GS})$. It holds the same role as $\Delta \mathrm{BIC}$, but works at low $S/N$. It requires the full $\chi^2$ of LePHARE.
    - 3. Star-PDF. It does not classify if a source is either a star or a galaxy directly, but if it is a star or not. It requires the full $\chi^2$ of LePHARE, but from star SEDs only.

## **0. Initialization**

### 0.1 Packages

In [1]:
#--- Classics ---
import re
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

#--- Stats & Maths---
from scipy.special import logsumexp
from scipy.stats import norm

In [2]:
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(base_dir)
from scripts import statsplot as lsp
from scripts import utils as sutils

### 0.2 Entries

In [3]:
# The output of zphota is mandatory for BIC method
dc2_BT_LSST_photoz = lsp.lephare_to_pandas(f"{base_dir}/simulation_catalogs/DC2/dp02_dc2_62_BT_LSST_full.out")
photoz_cat = dc2_BT_LSST_photoz

# FullLib for Bayes method


In [4]:
# --- Keep what is useful ---

photoz_cat= photoz_cat[['IDENT','CHI_BEST', 'CHI_STAR', 
                            'MAG_OBS0', 'MAG_OBS1', 'MAG_OBS2', 'MAG_OBS3', 'MAG_OBS4', 'MAG_OBS5',
                            'ERR_MAG_OBS0', 'ERR_MAG_OBS1', 'ERR_MAG_OBS2', 'ERR_MAG_OBS3', 'ERR_MAG_OBS4', 'ERR_MAG_OBS5', 'ZSPEC']].copy()

# ZSPEC=0 -> star
photoz_cat["IS_STAR"] = (photoz_cat["ZSPEC"] == 0).astype(int)
photoz_cat[:5]

,IDENT,CHI_BEST,CHI_STAR,MAG_OBS0,MAG_OBS1,MAG_OBS2,MAG_OBS3,MAG_OBS4,MAG_OBS5,ERR_MAG_OBS0,ERR_MAG_OBS1,ERR_MAG_OBS2,ERR_MAG_OBS3,ERR_MAG_OBS4,ERR_MAG_OBS5,ZSPEC,IS_STAR
0,1651413688361421449,2.52301,5.542650,26.112,26.669,25.962,25.452,25.985,24.877,0.445,0.249,0.155,0.191,0.823,0.523,0.00000,1
1,1651413688361451863,5.76531,19.808500,28.114,25.847,25.458,24.611,24.224,24.009,2.443,0.108,0.091,0.082,0.145,0.210,0.00000,1
2,1651413688361451723,7.65501,6.923500,27.572,26.454,27.334,26.706,26.399,25.543,1.066,0.125,0.343,0.362,0.781,0.741,1.58383,0
3,1651413688361451008,5.86742,4.896040,27.707,26.642,27.106,27.495,25.900,25.414,1.268,0.159,0.302,0.793,0.519,0.667,1.57419,0
4,1651413688361450766,1.59516,0.500102,28.423,26.166,25.975,25.918,26.159,26.046,3.301,0.140,0.140,0.254,0.835,1.465,1.40663,0


In [5]:
#--- Algorithms to use ---
use_BIC = True
use_bayes = False
use_starpdf = True
probability_threshold = 0.95 # [0,1] If specified, whenever the probability threshold has been crossed, 
                             # stop the algorithm for the current source at the current probability result.
                             # to choose after computing ROC/RP curves 

#--- Parameters ---
kG = 3 # dof of model A. Unknow, Z, alpha (normalisation)
kS = 4 # dof of model B. Teff, logg, FeH, alpha (normalisation)
nbands = 6   # ugrizy

#--- Preferences ---
ignore_aberrant_Z = True # sometimes best-likelihood can't be compute during photo-z
                         # yet it returns Z = -99


## **1. $\chi^2_{best fit}$ Comparison**

In [6]:
def BIC(chi2, k, n):
    return chi2 + k*np.log(n)

def AB_probability_from_bic(BIC_A, BIC_B, prior_A=0.5, prior_B=0.5):
    wA = prior_A * np.exp(-0.5 * BIC_A)
    wB = prior_B * np.exp(-0.5 * BIC_B)

    norm = wA + wB
    return wA / norm, wB / norm

In [7]:
BICg = BIC(40,kG,nbands)
BICs = BIC(47,kS,nbands)
AB_probability_from_bic(BICg, BICs)

(np.float64(0.9878220993676216), np.float64(0.012177900632378405))

## **2. Bayes Factor**

In [8]:
### We will implement it in last
# Not sure because it is computationally heavy

## **3. Star-PDF**

The functions come from the SPLIT_src.py

In [9]:
from scripts.split_src import SED_GRID, STAR_PDF, STAR_PDF_ANALYZER, SAMPLE_ANALYZER, SPLIT_TRAINING, SPLIT

Run this part if the training table is not ready

In [10]:
# # --- SED grid & star-PDF ---
# chi2_values_path = os.path.join(base_dir, 'simulation_catalogs/DC2/dp02_dc2_62_BT_LSST_full_PDFstar.prob')
# sed_grid_path = os.path.join(base_dir, 'simulated_star_sed/bt_spectra/bt_star_sed_full.list')
# sample_analyzer = SAMPLE_ANALYZER(chi2_values_path, sed_grid_path, nrows=1000000)

# training_sample_features = sample_analyzer.stat_feature_distribution() # the statistical features for each source
# best_fit_distribution = sample_analyzer.best_fit_distribution() # the best physical quantities computed for each source

# Let's make it one and keep the interesting features. (colnames: best_fit_distribution.columns)
# training_sample = sutils.join_tables(training_sample_features, [(dp02_dc2_62_table[['mt_match_objectId', 'ts_truth_type', 'imag']], 'source_id', 'mt_match_objectId'), 
#         (best_fit_distribution[['source_id', 'best_sed_id', 'Teff_best', 'logg_best', 'FeH_best', 'chi2_best']], 'source_id', 'source_id')], 
#         how='left')

Else just import the training table

In [ ]:
### The initial table, used in LePHARE
dp02_dc2_62_table = pd.read_csv(f'{base_dir}/simulation_catalogs/DC2/dp02_dc2_62.csv')
### The table in which is gathered the statistics compute from each onesource
training_sample_features = pd.read_csv(f'{base_dir}/simulation_catalogs/star_gal/training_sample_features.csv')
### The table of best fit statistical parameters
best_fit_distribution = pd.read_csv(f'{base_dir}/simulation_catalogs/star_gal/best_fit_distribution.csv')

# Let's make it one and keep the interesting features. (colnames: best_fit_distribution.columns)
training_sample = sutils.join_tables(training_sample_features, [(dp02_dc2_62_table[['mt_match_objectId', 'ts_truth_type', 'imag']], 'source_id', 'mt_match_objectId'), 
        (best_fit_distribution[['source_id', 'best_sed_id', 'Teff_best', 'logg_best', 'FeH_best', 'chi2_best']], 'source_id', 'source_id')], 
        how='left')

In [ ]:
# Some quantities dertemined with LePHARE are discrete, hence we can set up list of bins to work with to gain efficientcy
teff_values = list(range(800, 7200, 400)) + list(range(7200, 12001, 600)) + list(range(12500, 25001, 5000)) + list(range(25000, 50001, 5000))
logg_values = np.arange(0.0, 5.6, 0.5)
feh_values = [-3, -2, -1, 0.0, 0.3, 0.5]
sed_id_bins = np.arange(0,1622,1)

In [ ]:
split_training=SPLIT_TRAINING()

pdf_dict = split_training.pdfs_from_sample(
    training_sample,
    label_column="ts_truth_type",
    scols=["source_id", "npeaks_Teff", "npeaks_logg", 'npeaks_FeH', 'mt_match_objectId', 'ts_truth_type', 'chi2_best'],
    compute_weight=True,
    weight_method="f_classif",
    exclude_scols=True,
    selectKbest=None,
    bins=1000,
    interp=False,
    custo_bins_interp=[("Teff_best", teff_values, False),("logg_best", logg_values, False),("FeH_best", feh_values, False),('best_sed_id', sed_id_bins,None)],
    recenter_bins=True,
    normalize=True
)

## **4. Main**

In [ ]:
if ignore_aberrant_Z == True:
    photoz_cat = photoz_cat[photoz_cat['CHI_BEST']<1e9]

P_STAR_BIC = []

#--- BIC ---
if use_BIC == True:
    for source in photoz_cat.itertuples():
        chiG = source.CHI_BEST
        chiS = source.CHI_STAR
        BIC_G = BIC(chiG, kG, nbands)
        BIC_S = BIC(chiS, kS, nbands)
        pBIC_Gx, pBIC_Sx = AB_probability_from_bic(BIC_G, BIC_S)#, prior_A=0.9,prior_B=0.1)
        P_STAR_BIC.append(pBIC_Sx)

photoz_cat["P_STAR_BIC"] = P_STAR_BIC

photoz_cat_badPstar = photoz_cat[
        photoz_cat["P_STAR_BIC"]<probability_threshold # continue for the sources with not satisfying PSTAR
        ]

#--- PDF Star ---
split = SPLIT(pdf_dict)
probs = split.classify_sample(photoz_cat_badPstar, priors=None, use_weights=True)
final = pd.concat([photoz_cat_badPstar, probs], axis=1)



/tmp/ipykernel_7833/1491331496.py:9: RuntimeWarning: invalid value encountered in scalar divide
  return wA / norm, wB / norm


In [16]:
photoz_cat[(photoz_cat["P_STAR"]<0.9)&(photoz_cat["IS_STAR"]==1)]

,IDENT,CHI_BEST,CHI_STAR,MAG_OBS0,MAG_OBS1,MAG_OBS2,MAG_OBS3,MAG_OBS4,MAG_OBS5,ERR_MAG_OBS0,ERR_MAG_OBS1,ERR_MAG_OBS2,ERR_MAG_OBS3,ERR_MAG_OBS4,ERR_MAG_OBS5,ZSPEC,IS_STAR,P_STAR
0,1651413688361421449,2.523010,5.54265,26.112,26.669,25.962,25.452,25.985,24.877,0.445,0.249,0.155,0.191,0.823,0.523,0.0,1,0.082739
1,1651413688361451863,5.765310,19.80850,28.114,25.847,25.458,24.611,24.224,24.009,2.443,0.108,0.091,0.082,0.145,0.210,0.0,1,0.000364
428,1651413688361420917,3.512750,2.26498,26.938,28.655,27.654,25.460,24.452,23.838,0.688,1.113,0.508,0.127,0.143,0.155,0.0,1,0.432417
588,1651413688361453532,4.124040,17.77180,23.953,23.180,22.723,22.519,22.430,22.371,0.044,0.008,0.006,0.009,0.021,0.039,0.0,1,0.000444
795,1651413688361452730,1.381180,3.24882,29.716,27.118,26.371,25.203,24.886,24.816,8.376,0.238,0.150,0.094,0.202,0.372,0.0,1,0.138274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99049,1567947561673723026,1.777060,2.61327,27.334,27.544,26.424,25.931,26.135,25.017,0.894,0.333,0.148,0.175,0.617,0.470,0.0,1,0.211821
99199,1567947561673724335,5.540640,7.02821,29.965,25.159,24.300,23.914,23.610,23.584,10.418,0.040,0.022,0.028,0.063,0.129,0.0,1,0.162511
99328,1567947561673725364,40.662700,49.71070,23.347,22.534,22.256,22.130,22.159,22.070,0.028,0.004,0.004,0.006,0.017,0.032,0.0,1,0.004408
99780,1567947561673720563,0.102626,2.19963,29.001,27.494,26.980,26.212,25.844,25.221,4.116,0.320,0.241,0.226,0.482,0.594,0.0,1,0.125167
